In [1]:
!pip install sentence-transformers faiss-cpu networkx transformers

In [2]:
import numpy as np
import pandas as pd
import faiss
import pickle
import networkx as nx
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
final_df = pd.read_pickle("rag_data.pkl")

embeddings = np.load("embeddings.npy")

index = faiss.read_index("faiss_index.idx")

with open("kg_final.pkl", "rb") as f:
    G = pickle.load(f)

with open("centroids.pkl", "rb") as f:
    cluster_centroids = pickle.load(f)    
    



In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12420.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13948.69it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 10066.16it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present i

In [5]:
cache = {}

In [6]:
def rewrite_query_llm(query):
    
    prompt = f"Rewrite this agricultural query in clear English: {query}"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
def embed(query):
    return model.encode([query], normalize_embeddings=True)

In [8]:
def faiss_retrieve(query, k=5):
    
    q_emb = embed(query)
    
    D, I = index.search(q_emb, k)
    
    return final_df.iloc[I[0]]

In [9]:
def kg_retrieve(query, cluster_centroids):
    
    q_emb = embed(query)
    
    best_cluster = None
    best_sim = -1
    
    for c, centroid in cluster_centroids.items():
        
        sim = (q_emb @ centroid.reshape(-1,1))[0][0]
        
        if sim > best_sim:
            best_sim = sim
            best_cluster = c
    
    node = f"cluster_{best_cluster}"
    
    answers = []
    
    if node in G:
        for n in G.neighbors(node):
            if G.nodes[n]["type"] == "answer":
                answers.append(n)
    
    return answers

In [10]:
def merge_results(faiss_df, kg_results):
    
    faiss_answers = faiss_df["clean_answer"].tolist()
    
    return list(set(faiss_answers + kg_results))

In [11]:
def rerank(query, candidates, k=5):
    
    pairs = [[query, c] for c in candidates]
    
    scores = cross_encoder.predict(pairs)
    
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    return [x[0] for x in ranked[:k]]

In [12]:
def build_context(results):
    return "\n".join(results)

In [13]:
def filter_answers(query, answers):
    
    query_words = set(query.split())
    
    filtered = []
    
    for ans in answers:
        if any(word in ans for word in query_words):
            filtered.append(ans)
    
    return filtered

In [14]:
def clean_answers(answers):
    
    cleaned = []
    
    for ans in answers:
        ans = ans.replace("and", ",")
        ans = ans.strip()
        cleaned.append(ans)
    
    return cleaned

In [37]:
def generate_answer(query, answers):

    # 1. filter + clean
    answers = filter_answers(query, answers)
    answers = clean_answers(answers)

    if len(answers) == 0:
        return "Not enough information. Please provide more details."

    # 2. take top 3
    answers = answers[:3]

    # 3. build clean context (NO labels, NO numbering)
    context = "\n".join(answers)

    # 4. STRONG CONTROLLED PROMPT
    prompt = f"""
Rewrite the following agricultural recommendations into a clean, well-structured answer.

STRICT RULES:
- Do NOT repeat the query
- Do NOT include words like "treatments", "data", or "information"
- Do NOT add new information
- Fix spelling and grammar
- Combine similar points if needed
- Output ONLY final answer in bullet points

Content:
{context}
"""

    # 5. tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model_llm.device) for k, v in inputs.items()}

    # 6. generate (deterministic)
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 7. FINAL CLEANUP (VERY IMPORTANT)
    result = result.replace(prompt, "").strip()

    # remove any accidental leakage
    result = result.replace("Content:", "")
    result = result.replace("Answer:", "")
    result = result.replace(query, "")

    return result.strip()

In [38]:
def run_pipeline(query, cluster_centroids, k=5):

    # 1. rewrite
    rewritten = rewrite_query_llm(query)

    # 2. retrieval
    faiss_res = faiss_retrieve(rewritten, k)
    kg_res = kg_retrieve(rewritten, cluster_centroids)

    # 3. merge
    combined = merge_results(faiss_res, kg_res)

    # 4. rerank
    final_answers = rerank(rewritten, combined, k)

    # 5. final answer (NO context building separately)
    answer = generate_answer(rewritten, final_answers)

    return {
        "original_query": query,
        "rewritten_query": rewritten,
        "answers": final_answers,
        "final_answer": answer
    }

In [41]:
output = run_pipeline("yellowing of leaf in paddy ", cluster_centroids)



In [42]:
print("\n🔹 Rewritten Query:")
print(output["rewritten_query"])

print("\n🔹 Retrieved Answers:")
for i, ans in enumerate(output["answers"], 1):
    print(f"{i}. {ans}")

print("\n🔹 Final Answer:")
print(output["final_answer"])


🔹 Rewritten Query:
Yellowing of leaf in paddy

🔹 Retrieved Answers:
1. information regarding control of yellow of leaf in paddyspray 25 kilogram urea 500 gram zinc in 100 liter of water acre
2. yellow colour in paddydhan spray in copper oxichloride 2 gmwater
3. yellow discoloration of paddyspray immidaclopid 1 milliliter per 3 liter water
4. if yellow is occurring in old leaves it is due to nitrogen deficiency if it is young leaves that may be due to iron deficiency in rice foliar spraying of 125 gramkanal of ferrous sulphate dissolved in 125 liter of waterafter 3 4 days spray mancozeb 25 gramper liter of water
5. to control yellow colour of leaves in rice spray ferrous sulphate 1 kgacre with 100 litter water

🔹 Final Answer:
urea, zinc, and copper oxychloride are recommended for controlling yellow of leaf in paddyspray.


In [47]:
output = run_pipeline("too much water in my field ", cluster_centroids)

In [48]:
print("\n🔹 Rewritten Query:")
print(output["rewritten_query"])

print("\n🔹 Retrieved Answers:")
for i, ans in enumerate(output["answers"], 1):
    print(f"{i}. {ans}")

print("\n🔹 Final Answer:")
print(output["final_answer"])


🔹 Rewritten Query:
Is there too much water in my field?

🔹 Retrieved Answers:
1. drain out excess water from field
2. recommended eradicate water from field opt for help from aao
3. rogor 250 milliliter 200 liter water per acre
4. kisan ji cloropayariphas 1 liter dawa acre ke hisab se sichai ke water ke sath field me pahucha de
5. recommended to spray saaf carbendazim 12 mancozeb 63 400 gram per 200 liter of water per acre 3 gram per 1 litrerecommended to spray plantomycin 200 gram in 200 liter of water per acre of land weather there is possibilities of medium to light amount of rainfall in next 5 days and cloudy weather will remain

🔹 Final Answer:
rogor 250 milliliter 200 liter water per acre
